# Imports

In [ ]:
import multisesh as ms
import pandas as pd
import numpy as np

# Load data and set-up

### names to set

In [26]:
root = "" # path of the dataset
seg_out_multi = ''  # path to save segmentations to 
df_out = '' # path to save full dataset measurements to
norm_intensity_out = '' # path to save the final normalised measures to

### load data set

In [8]:
xfold = ms.XFold(root,MustNOTContain=['DAPI'],MustContainAND=['pfak.'])

xfold.Summarise()

Total no. of sessions: 38
Total no. of tiff files: 38
Total memory of tiff files: 79.759843 MB
Total no. of time points (according to metadata): 38
Total no. of fields (no. of unique ID): 1
Total time span: 41 days, 15 hours, 0 minutes.

The following shows only the value of the given attribute 
when it changes from one session to the next: 
Time points: [1]
Fields: [1]
Montage tiles: [1]
z-Slices: [1]
number of channels: [1]
ZStep: [1] um
names of channels: [['Channel:0:0']]

The names of the sessions: 
Replicate1->CTRL->stiff_CTRL_pFAK-Phall-ZO-1_colony 1_pfak
Replicate1->CTRL->stiff_CTRL_pFAK-Phall-ZO-1_colony 2_pfak
Replicate1->CTRL->stiff_CTRL_pFAK-Phall-ZO-1_colony 3_pfak
Replicate1->CTRL->stiff_CTRL_pFAK-Phall-Zo1_colony 4_pfak
Replicate1->CTRL->stiff_CTRL_pFAK-Phall-Zo1_colony 5_pfak
Replicate1->CTRL->stiff_CTRL_pFAK-Phall-Zo1_colony 6_pfak
Replicate1->CTRL->stiff_CTRL_pFAK-Phall-Zo1_colony 7_pfak
Replicate1->CTRL->stiff_CTRL_pFAK-Phall-Zo1_colony 8_pfak
Replicate1->LAM->stiff_

# Apply multi-otsu threshold to whole data set

In [11]:
xfold.Threshold(0,blur=1,method='multi-otsu',saveSegmentations=seg_out_multi,removeSmall=15,clearBorder=True)

# Visualise thresholds

In [12]:
xfold_seg = ms.XFold(seg_out_multi,OriginalXFold=xfold)

In [13]:
tdata = xfold.makeTData(S=14)

tdata.Plot(plotSegMask=seg_out_multi)

Output()

# Measure properties

In [15]:
fun1 = [[ms.RegionPropFunctions.measure_channel_intensity,[0,seg_out_multi],[]]]

df = xfold.RegionProps(Segmentation=seg_out_multi,saveDF=df_out,fun=fun1)

# Add condition columns

In [3]:
df['condition'] = df['Session_Name'].str.extract(r"_(CTRL|LAM)_")

# Make normalised measures and save dataframe

In [21]:
mean_col_multi = '' # put column name of column you want to normalise here, should look like: mean_of_Channel:0:0_in_Segmentation_...'
mean_foc = df.groupby(["condition", "Session_Name"])[mean_col_multi].mean().reset_index(name="count")
mean_foc['count normalised'] = mean_foc['count']/np.max(mean_foc['count'])

mean_foc.to_csv(norm_intensity_out)